# Chapitre 5 - Exercices 5.7 à 5.10

Ce notebook traite les exercices sur la réplication multi-leader, sans leader et la
détection des écritures concurrentes :

- **5.7** : conflits en multi-leader : horloges, identifiants et appareils hors ligne
  (calcul et simulation) ;
- **5.8** : choisir une topologie multi-leader (raisonnement et simulation) ;
- **5.9** : choisir n, w et r (calcul et simulation) ;
- **5.10** : vecteurs de versions à la main (raisonnement et mini-lab).

In [1]:
import itertools
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import replication as rp
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)

## Exercice 5.7 : conflits en multi-leader

**Énoncé.**

1. *Horloges.* Deux leaders résolvent les conflits par LWW avec leurs horloges physiques,
   synchronisées par NTP à ±20 ms près (décalage uniforme dans cet intervalle, indépendant
   pour chaque leader). Un utilisateur modifie son profil sur le leader A puis, 30 ms plus
   tard, le corrige depuis un autre appareil routé vers le leader B. Quelle est la
   probabilité que sa correction soit jetée ?
2. *Identifiants.* Trois leaders attribuent des identifiants auto-incrémentés. Proposer
   un schéma sans collision. Que se passe-t-il si l'on ajoute un quatrième leader ?
3. *Appareils hors ligne.* Un agenda partagé est modifié depuis 3 appareils ; chacun
   reste en moyenne 8 h hors ligne entre deux synchronisations ; un événement donné est
   modifié 2 fois par jour (réparties au hasard entre les appareils). Estimer la
   probabilité qu'une modification soit en conflit.

**Rappels.** LWW et horloges, évitement des conflits par partition de l'espace des
identifiants, conflits $\approx 1 - e^{-\lambda L}$ (cours 3, sections 4 à 6, sections
*Handling Write Conflicts* et *Last write wins*).

**Solution.**

*1.* La correction (horodatage $30 + \varepsilon_B$) perd si $30 + \varepsilon_B <
\varepsilon_A$, soit $\varepsilon_A - \varepsilon_B > 30$ ms. La différence de deux
uniformes sur $[-20, 20]$ suit une loi triangulaire sur $[-40, 40]$ ;
$P(D > 30) = \frac{(40 - 30)^2}{2 \times 40^2} = 3{,}125\,\%$. Une correction sur 32 est
silencieusement annulée.

*2.* Le leader $i \in \{0, 1, 2\}$ attribue $i, i + 3, i + 6, \dots$ (congrus à $i$
modulo 3). Ajouter un quatrième leader casse le schéma : passer au modulo 4 en repartant
de zéro réattribue des identifiants déjà émis (le leader 0 émettrait 4, 8, 12..., déjà
donnés par les leaders 1, 2 et 0 au modulo 3).
Il faut repartir d'un seuil commun (tous les nouveaux identifiants au-delà du maximum
déjà émis), réserver dès le départ un modulo plus grand que nécessaire, ou utiliser des
identifiants (préfixe du leader, compteur) ou des UUID.

*3.* Une modification k n'est vue d'un autre appareil b qu'après la synchronisation de
l'auteur **puis** celle de b ; avec des intervalles exponentiels (sans mémoire), chacune
arrive en moyenne 8 h plus tard : k reste invisible pour b environ 16 h. Une modification
de b est concurrente de k si elle tombe dans les 16 h qui suivent k (b ne connaît pas
encore k) ou dans les 16 h qui précèdent (k ignorait celle de b) : fenêtre d'environ
32 h. Les modifications des **autres** appareils arrivent au rythme
$\lambda \approx 2 \times 2/3 = 1{,}33$ par jour. D'où
$P \approx 1 - e^{-1{,}33 \times 32/24} \approx 83\,\%$ : un ordre de grandeur (les
fenêtres ne sont pas fixes), que la simulation affine. Le message est clair : hors ligne,
une donnée partagée modifiée chaque jour est presque toujours en conflit ; il faut une
fusion automatique.

**Vérification.**

In [2]:
N = 1_000_000
eA, eB = rng.uniform(-20, 20, N), rng.uniform(-20, 20, N)
print(f"1. correction jetée : simulation {100 * np.mean(30 + eB < eA):.2f} %, formule {100 * 10**2 / (2 * 40**2):.3f} %")

def ids(i, k, n, debut=0):
    return {debut + i + k * j for j in range(n)}
anciens = set().union(*(ids(i, 3, 5) for i in range(3)))
nouveaux_naifs = set().union(*(ids(i, 4, 5) for i in range(4)))
nouveaux_seuil = set().union(*(ids(i, 4, 5, debut=max(anciens) + 1) for i in range(4)))
print("2. collisions modulo 4 naïf :", sorted(anciens & nouveaux_naifs), "| avec seuil :", sorted(anciens & nouveaux_seuil))

def agenda(jours=2000, n_app=3, hors_ligne_h=8.0, modifs_par_jour=2.0, seed=1):
    rnd = np.random.default_rng(seed)
    T = jours * 24.0
    syncs = [np.cumsum(rnd.exponential(hors_ligne_h, int(T / hors_ligne_h * 2))) for _ in range(n_app)]
    t_mod = np.sort(rnd.uniform(0, T, rnd.poisson(modifs_par_jour * jours)))
    app = rnd.integers(0, n_app, len(t_mod))
    def prochaine(a, t):
        s = syncs[a]
        return s[np.searchsorted(s, t)]
    def precedente(a, t):
        s = syncs[a]; k = np.searchsorted(s, t) - 1
        return s[k] if k >= 0 else 0.0
    # une modification m est connue de l'appareil b à partir de max(sync de l'auteur après m, puis sync de b après)
    visible = [{b: (prochaine(b, prochaine(app[k], t)) if b != app[k] else t) for b in range(n_app)}
               for k, t in enumerate(t_mod)]
    conflit = np.zeros(len(t_mod), bool)
    for k in range(len(t_mod)):
        for j in range(k + 1, len(t_mod)):
            if t_mod[j] > t_mod[k] + 72:
                break
            if app[j] != app[k] and visible[k][app[j]] > t_mod[j]:
                conflit[k] = conflit[j] = True            # j ignorait k : écritures concurrentes
    return conflit.mean()

print(f"3. modifications en conflit (simulation) : {100 * agenda():.0f} %  ; estimation : {100 * (1 - math.exp(-4 / 3 * 32 / 24)):.0f} %")

1. correction jetée : simulation 3.13 %, formule 3.125 %
2. collisions modulo 4 naïf : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14] | avec seuil : []
3. modifications en conflit (simulation) : 78 %  ; estimation : 83 %


La simulation de l'agenda tient compte du fait que l'auteur et le lecteur doivent tous
deux se synchroniser ; elle confirme l'ordre de grandeur : plus de trois modifications
sur quatre sont concurrentes d'une autre.

> **À retenir.** Avec des horloges physiques, LWW jette aussi des écritures successives ;
> partitionner l'espace des identifiants évite les collisions tant que le nombre de
> leaders ne change pas ; hors ligne, les conflits sont la règle et non l'exception.